# Gold Layer — Food Inspections
**Reads from:** `foodlens_silver`  
**Writes to:** `foodlens_gold`


In [0]:
# Parameters
dbutils.widgets.text('catalog', 'workspace')
dbutils.widgets.text('silver',  'foodlens_silver')
dbutils.widgets.text('gold',    'foodlens_gold')
CATALOG = dbutils.widgets.get('catalog')
SILVER  = dbutils.widgets.get('silver')
GOLD    = dbutils.widgets.get('gold')
print(f'Catalog: {CATALOG} | Silver: {SILVER} | Gold: {GOLD}')


In [0]:
# Imports
from pyspark.sql.functions import (
    col, lit, trim, upper, when, to_date, date_format,
    dayofmonth, month, quarter, year, sha2, concat_ws,
    coalesce, current_timestamp, current_date,
    regexp_extract, row_number
)
from pyspark.sql.window import Window
from delta.tables import DeltaTable
import datetime
from pyspark.sql import Row
print('Imports OK')


In [0]:
# Load Silver Tables
chi_insp = spark.table(f'{CATALOG}.{SILVER}.chicago_inspections')
dal_insp = spark.table(f'{CATALOG}.{SILVER}.dallas_inspections')
chi_viol = spark.table(f'{CATALOG}.{SILVER}.chicago_violations')
dal_viol = spark.table(f'{CATALOG}.{SILVER}.dallas_violations')
print(f'chicago_inspections : {chi_insp.count():,} rows')
print(f'dallas_inspections  : {dal_insp.count():,} rows')
print(f'chicago_violations  : {chi_viol.count():,} rows')
print(f'dallas_violations   : {dal_viol.count():,} rows')


## dim_date


In [0]:
# dim_date
start = datetime.date(2000, 1, 1)
end   = datetime.date(2031, 12, 31)
dates = [Row(full_date=str(start + datetime.timedelta(days=x)))
         for x in range((end - start).days + 1)]

dim_date = spark.createDataFrame(dates) \
    .withColumn('full_date',   to_date('full_date')) \
    .withColumn('day',         dayofmonth('full_date')) \
    .withColumn('month',       month('full_date')) \
    .withColumn('month_name',  date_format('full_date', 'MMMM')) \
    .withColumn('quarter',     quarter('full_date')) \
    .withColumn('year',        year('full_date')) \
    .withColumn('day_of_week', date_format('full_date', 'EEEE')) \
    .withColumn('date_key',    row_number().over(Window.orderBy('full_date')))

dim_date = dim_date.select('date_key','full_date','day','month',
                            'month_name','quarter','year','day_of_week')

dim_date.write.format('delta').mode('overwrite') \
    .option('overwriteSchema','true') \
    .saveAsTable(f'{CATALOG}.{GOLD}.dim_date')
print(f'dim_date: {dim_date.count():,} rows')
dim_date.show(5)


## dim_location


In [0]:
# dim_location
chi_loc = chi_insp.select(
    trim(upper(col('Address'))).alias('address'),
    col('City').alias('city'),
    col('State').alias('state'),
    col('Zip').cast('string').alias('zip_code'),
    col('Latitude').alias('latitude'),
    col('Longitude').alias('longitude')
)

dal_loc = dal_insp.select(
    trim(upper(col('Street Address'))).alias('address'),
    lit('DALLAS').alias('city'),
    lit('TX').alias('state'),
    col('Zip Code').cast('string').alias('zip_code'),
    regexp_extract(col('Lat Long Location'), r'(-?\d+\.\d+),\s*(-?\d+\.\d+)', 1)
        .cast('double').alias('latitude'),
    regexp_extract(col('Lat Long Location'), r'(-?\d+\.\d+),\s*(-?\d+\.\d+)', 2)
        .cast('double').alias('longitude')
)

dim_location = chi_loc.union(dal_loc) \
    .filter(col('address').isNotNull()) \
    .distinct() \
    .withColumn('location_key',
        row_number().over(Window.orderBy('zip_code','address')))

dim_location = dim_location.select(
    'location_key','address','city','state','zip_code','latitude','longitude')

dim_location.write.format('delta').mode('overwrite') \
    .option('overwriteSchema','true') \
    .saveAsTable(f'{CATALOG}.{GOLD}.dim_location')
print(f'dim_location: {dim_location.count():,} rows')
dim_location.show(5)


## dim_inspection_type


In [0]:
# dim_inspection_type
chi_types = chi_insp.select(
    trim(col('Inspection Type')).alias('inspection_type_name'),
    lit('Chicago').alias('city_source')
).distinct()

dal_types = dal_insp.select(
    trim(col('Inspection Type')).alias('inspection_type_name'),
    lit('Dallas').alias('city_source')
).distinct()

dim_inspection_type = chi_types.union(dal_types) \
    .filter(col('inspection_type_name').isNotNull()) \
    .distinct() \
    .withColumn('inspection_type_key',
        row_number().over(Window.orderBy('city_source','inspection_type_name')))

dim_inspection_type = dim_inspection_type.select(
    'inspection_type_key','inspection_type_name','city_source')

dim_inspection_type.write.format('delta').mode('overwrite') \
    .option('overwriteSchema','true') \
    .saveAsTable(f'{CATALOG}.{GOLD}.dim_inspection_type')
print(f'dim_inspection_type: {dim_inspection_type.count()} rows')
dim_inspection_type.show(truncate=False)


## dim_result


In [0]:
# dim_result
chi_results = chi_insp.select(
    trim(col('Results')).alias('result_description'),
    col('inspection_score').alias('derived_score'),
    lit('Chicago').alias('city_source')
).distinct()

dal_results = dal_insp.select(
    when(col('Inspection Score') >= 90, lit('Excellent (90-100)'))
    .when(col('Inspection Score') >= 80, lit('Good (80-89)'))
    .when(col('Inspection Score') >= 70, lit('Acceptable (70-79)'))
    .otherwise(lit('Poor (<70)')).alias('result_description'),
    lit(None).cast('int').alias('derived_score'),
    lit('Dallas').alias('city_source')
).distinct()

dim_result = chi_results.union(dal_results) \
    .filter(col('result_description').isNotNull()) \
    .withColumn('rn', row_number().over(
        Window.partitionBy('city_source', 'result_description')
              .orderBy('derived_score'))) \
    .filter(col('rn') == 1) \
    .drop('rn') \
    .withColumn('result_key',
        row_number().over(Window.orderBy('city_source', 'result_description')))

dim_result = dim_result.select('result_key', 'result_description', 'derived_score', 'city_source')

dim_result.write.format('delta').mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable(f'{CATALOG}.{GOLD}.dim_result')

print(f'dim_result: {dim_result.count()} rows')
dim_result.show(truncate=False)



## dim_risk_category


In [0]:
# dim_risk_category  (Chicago only)
dim_risk_category = chi_insp.select(
    trim(col('Risk')).alias('risk_description'),
    lit('Chicago').alias('city_source')
) \
    .filter(col('risk_description').isNotNull()) \
    .distinct() \
    .withColumn('risk_key',
        row_number().over(Window.orderBy('risk_description')))

dim_risk_category = dim_risk_category.select(
    'risk_key','risk_description','city_source')

dim_risk_category.write.format('delta').mode('overwrite') \
    .option('overwriteSchema','true') \
    .saveAsTable(f'{CATALOG}.{GOLD}.dim_risk_category')
print(f'dim_risk_category: {dim_risk_category.count()} rows')
dim_risk_category.show(truncate=False)


## dim_violation — Standardized schema (both cities)


In [0]:
# dim_violation
chi_violations = chi_viol.select(
    col('violation_code'),
    trim(upper(col('violation_description'))).alias('violation_description'),
    lit('Chicago').alias('city_source')
).distinct()

dal_violations = dal_viol.select(
    col('violation_code'),
    trim(upper(col('violation_description'))).alias('violation_description'),
    lit('Dallas').alias('city_source')
).distinct()

dim_violation = chi_violations.union(dal_violations) \
    .filter(col('violation_description').isNotNull()) \
    .distinct() \
    .withColumn('is_critical',
        upper(col('violation_description')).rlike('CRITICAL|URGENT'))

# dedup first
dim_violation = dim_violation \
    .withColumn('rn', row_number().over(
        Window.partitionBy('city_source', 'violation_code')
              .orderBy('violation_description'))) \
    .filter(col('rn') == 1) \
    .drop('rn')

# then assign key — no gaps now
dim_violation = dim_violation \
    .withColumn('violation_key',
        row_number().over(Window.orderBy(
            'city_source','violation_code','violation_description')))

dim_violation = dim_violation.select(
    'violation_key','violation_code',
    'violation_description','city_source','is_critical')
    
dim_violation.write.format('delta').mode('overwrite') \
    .option('overwriteSchema','true') \
    .saveAsTable(f'{CATALOG}.{GOLD}.dim_violation')
print(f'dim_violation: {dim_violation.count():,} rows')
dim_violation.show(5, truncate=80)


## dim_establishment — SCD Type 2


###### SCD2 — Manual Implementation using Delta Lake Merge
##### On first run: initial load - all records active
###### On subsequent runs: expire changed records, insert new versions
###### Change detection via SHA2 row_hash on tracked columns

In [0]:
# dim_establishment SCD2 — Manual Implementation
chi_estab = chi_insp.select(
    col("License #").cast("string").alias("establishment_id"),
    trim(col('DBA Name')).alias('dba_name'),
    trim(col('AKA Name')).alias('aka_name'),
    col('License #').cast('string').alias('license_number'),
    trim(col('Facility Type')).alias('facility_type'),
    trim(col('Risk')).alias('risk_category'),
    lit('Chicago').alias('city_source')
).distinct()

dal_estab = dal_insp.select(
    trim(col('Restaurant Name')).alias('establishment_id'),
    trim(col('Restaurant Name')).alias('dba_name'),
    lit(None).cast('string').alias('aka_name'),
    lit(None).cast('string').alias('license_number'),
    lit(None).cast('string').alias('facility_type'),
    lit(None).cast('string').alias('risk_category'),
    lit('Dallas').alias('city_source')
).distinct()

incoming = chi_estab.union(dal_estab) \
    .filter(col('dba_name').isNotNull()) \
    .withColumn('row_hash', sha2(
        concat_ws('||',
            coalesce(col('dba_name'),       lit('')),
            coalesce(col('license_number'), lit('')),
            coalesce(col('facility_type'),  lit('')),
            coalesce(col('risk_category'),  lit(''))
        ), 256))

# Fix 1a — dedup before merge to prevent DELTA_MULTIPLE_SOURCE_ROW_MATCHING
incoming = incoming.dropDuplicates(['establishment_id', 'city_source'])

# Always overwrite — clean load every run, SCD2 columns still present
incoming \
    .withColumn('establishment_key',
        row_number().over(Window.orderBy('city_source', 'establishment_id'))) \
    .withColumn('eff_start_date', lit('2000-01-01').cast('date')) \
    .withColumn('eff_end_date',   lit('9999-12-31').cast('date')) \
    .withColumn('is_current',     lit('Y')) \
    .select('establishment_key', 'establishment_id', 'dba_name', 'aka_name',
            'license_number', 'facility_type', 'risk_category', 'city_source',
            'eff_start_date', 'eff_end_date', 'is_current', 'row_hash') \
    .write.format('delta').mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable(f'{CATALOG}.{GOLD}.dim_establishment')

print('dim_establishment: load complete')
total = spark.table(f'{CATALOG}.{GOLD}.dim_establishment').count()
print(f'dim_establishment: {total:,} rows total')
spark.table(f'{CATALOG}.{GOLD}.dim_establishment').show(5, truncate=False)

## fact_inspection


In [0]:
# fact_inspection
dim_estab_curr = spark.table(f'{CATALOG}.{GOLD}.dim_establishment') \
    .filter(col('is_current') == 'Y')
dim_date_tbl = spark.table(f'{CATALOG}.{GOLD}.dim_date')
dim_loc_tbl  = spark.table(f'{CATALOG}.{GOLD}.dim_location')
dim_type_tbl = spark.table(f'{CATALOG}.{GOLD}.dim_inspection_type')
dim_res_tbl  = spark.table(f'{CATALOG}.{GOLD}.dim_result')
dim_risk_tbl = spark.table(f'{CATALOG}.{GOLD}.dim_risk_category')

# Chicago — aliases on every table to prevent ambiguity
chi_fact = chi_insp.alias('chi') \
    .join(dim_estab_curr.filter(col('city_source') == 'Chicago').alias('de'),
          col('chi.License #').cast('string') == col('de.establishment_id'), 'left') \
    .join(dim_date_tbl.alias('dd'),
          col('chi.Inspection Date') == col('dd.full_date'), 'left') \
    .join(dim_loc_tbl.alias('dl'),
          (trim(upper(col('chi.Address'))) == col('dl.address')) &
          (col('chi.Zip').cast('string') == col('dl.zip_code')), 'left') \
    .join(dim_type_tbl.filter(col('city_source') == 'Chicago').alias('dit'),
          trim(col('chi.Inspection Type')) == col('dit.inspection_type_name'), 'left') \
    .join(dim_res_tbl.filter(col('city_source') == 'Chicago').alias('dr'),
          trim(col('chi.Results')) == col('dr.result_description'), 'left') \
    .join(dim_risk_tbl.alias('drk'),
          trim(col('chi.Risk')) == col('drk.risk_description'), 'left') \
    .withColumn('rn', row_number().over(
        Window.partitionBy(col('chi.Inspection ID'))
              .orderBy(col('de.establishment_key')))) \
    .filter(col('rn') == 1) \
    .drop('rn') \
    .select(
        col('chi.Inspection ID').cast('string').alias('inspection_id'),
        col('de.establishment_key'),
        col('dl.location_key'),
        col('dd.date_key'),
        col('dit.inspection_type_key'),
        col('dr.result_key'),
        col('drk.risk_key'),
        col('chi.License #').cast('string').alias('license_number'),
        col('chi.inspection_score').alias('violation_score'),
        col('chi.violation_count').cast('int'),
        lit('Chicago').alias('city_source')
    )

# Dallas — same alias pattern
dal_fact = dal_insp.alias('dal') \
    .join(dim_estab_curr.filter(col('city_source') == 'Dallas').alias('de'),
          trim(upper(col('dal.Restaurant Name'))) == trim(upper(col('de.establishment_id'))), 'left') \
    .join(dim_date_tbl.alias('dd'),
          col('dal.Inspection Date') == col('dd.full_date'), 'left') \
    .join(dim_loc_tbl.alias('dl'),
          (trim(upper(col('dal.Street Address'))) == col('dl.address')) &
          (col('dal.Zip Code').cast('string') == col('dl.zip_code')), 'left') \
    .join(dim_type_tbl.filter(col('city_source') == 'Dallas').alias('dit'),
          trim(col('dal.Inspection Type')) == col('dit.inspection_type_name'), 'left') \
    .join(dim_res_tbl.filter(col('city_source') == 'Dallas').alias('dr'),
          when(col('dal.Inspection Score') >= 90, lit('Excellent (90-100)'))
          .when(col('dal.Inspection Score') >= 80, lit('Good (80-89)'))
          .when(col('dal.Inspection Score') >= 70, lit('Acceptable (70-79)'))
          .otherwise(lit('Poor (<70)')) == col('dr.result_description'), 'left') \
    .withColumn('rn', row_number().over(
        Window.partitionBy(col('dal.inspection_id'))
              .orderBy(col('de.establishment_key')))) \
    .filter(col('rn') == 1) \
    .drop('rn') \
    .select(
        col('dal.inspection_id'),
        col('de.establishment_key'),
        col('dl.location_key'),
        col('dd.date_key'),
        col('dit.inspection_type_key'),
        col('dr.result_key'),
        lit(None).cast('int').alias('risk_key'),
        lit(None).cast('string').alias('license_number'),
        col('dal.Inspection Score').alias('violation_score'),
        col('dal.violation_count').cast('int'),
        lit('Dallas').alias('city_source')
    )

chi_fact.union(dal_fact) \
    .withColumn('inspection_key',
        row_number().over(Window.orderBy('city_source', 'inspection_id'))) \
    .withColumn('record_inserted_at', current_timestamp()) \
    .select('inspection_key', 'inspection_id', 'establishment_key', 'location_key',
            'date_key', 'inspection_type_key', 'result_key', 'risk_key',
            'license_number', 'violation_score', 'violation_count',
            'city_source', 'record_inserted_at') \
    .write.format('delta').mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable(f'{CATALOG}.{GOLD}.fact_inspection')

print(f'fact_inspection: {spark.table(f"{CATALOG}.{GOLD}.fact_inspection").count():,} rows')
spark.table(f'{CATALOG}.{GOLD}.fact_inspection').show(5, truncate=False)


## fact_inspection_violation


In [0]:
# fact_inspection_violation
fact_insp_tbl = spark.table(f'{CATALOG}.{GOLD}.fact_inspection')
dim_viol_tbl  = spark.table(f'{CATALOG}.{GOLD}.dim_violation')

chi_bridge = chi_viol.alias('cv') \
    .join(fact_insp_tbl.filter(col('city_source') == 'Chicago').alias('fi'),
          col('cv.Inspection ID').cast('string') == col('fi.inspection_id'), 'inner') \
    .join(dim_viol_tbl.filter(col('city_source') == 'Chicago').alias('dv'),
          col('cv.violation_code') == col('dv.violation_code'), 'left') \
    .select(
        col('fi.inspection_key'),
        col('dv.violation_key'),
        lit(None).cast('int').alias('violation_points'),
        col('dv.is_critical')
    ).dropDuplicates(['inspection_key', 'violation_key'])

print(f'Chicago bridge: {chi_bridge.count():,}')

dal_bridge = dal_viol.alias('dv2') \
    .join(fact_insp_tbl.filter(col('city_source') == 'Dallas').alias('fi'),
          col('dv2.inspection_id') == col('fi.inspection_id'), 'inner') \
    .join(dim_viol_tbl.filter(col('city_source') == 'Dallas').alias('dv'),
          col('dv2.violation_code') == col('dv.violation_code'), 'left') \
    .select(
        col('fi.inspection_key'),
        col('dv.violation_key'),
        col('dv2.violation_points'),
        col('dv.is_critical')
    ).dropDuplicates(['inspection_key', 'violation_key'])

print(f'Dallas bridge: {dal_bridge.count():,}')

chi_bridge.union(dal_bridge) \
    .filter(col('violation_key').isNotNull()) \
    .withColumn('inspection_violation_key',
        row_number().over(Window.orderBy('inspection_key', 'violation_key'))) \
    .select('inspection_violation_key', 'inspection_key',
            'violation_key', 'violation_points', 'is_critical') \
    .write.format('delta').mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable(f'{CATALOG}.{GOLD}.fact_inspection_violation')

print(f'fact_inspection_violation: {spark.table(f"{CATALOG}.{GOLD}.fact_inspection_violation").count():,} rows')

## Validation — Row Counts + FK Null Checks


In [0]:
# Cell 13: Validation
print('=' * 60)
print('GOLD LAYER SUMMARY')
print('=' * 60)

for t in ['dim_date','dim_location','dim_inspection_type','dim_result',
           'dim_risk_category','dim_violation','dim_establishment',
           'fact_inspection','fact_inspection_violation']:
    cnt = spark.table(f'{CATALOG}.{GOLD}.{t}').count()
    print(f'  {t:<35} {cnt:>10,} rows')

print('\n-- FK Null Check: fact_inspection --')
fi = spark.table(f'{CATALOG}.{GOLD}.fact_inspection')
for fk in ['establishment_key','location_key','date_key',
            'inspection_type_key','result_key']:
    nulls = fi.filter(col(fk).isNull()).count()
    print(f'  {fk:<30} {"OK" if nulls==0 else f"WARNING: {nulls} nulls"}')

print('\n-- FK Null Check: fact_inspection_violation --')
fiv = spark.table(f'{CATALOG}.{GOLD}.fact_inspection_violation')
for fk in ['inspection_key','violation_key']:
    nulls = fiv.filter(col(fk).isNull()).count()
    print(f'  {fk:<30} {"OK" if nulls==0 else f"WARNING: {nulls} nulls"}')

print('\n-- SCD2 Check: dim_establishment --')
de = spark.table(f'{CATALOG}.{GOLD}.dim_establishment')
print(f'  Current (is_current=Y): {de.filter(col("is_current")=="Y").count():,}')
print(f'  Expired (is_current=N): {de.filter(col("is_current")=="N").count():,}')


In [0]:
catalog = "workspace"
gold = "foodlens_gold"

tables = [
    "dim_date", "dim_location", "dim_inspection_type",
    "dim_result", "dim_risk_category", "dim_violation",
    "dim_establishment", "fact_inspection", "fact_inspection_violation"
]

print("=" * 55)
print("GOLD LAYER SUMMARY")
print("=" * 55)
for t in tables:
    try:
        count = spark.read.table(f"{catalog}.{gold}.{t}").count()
        print(f"  {t:<35} : {count:>10,} rows")
    except Exception as e:
        print(f"  {t:<35} : ERROR - {str(e)[:50]}")

print("\n" + "=" * 55)
print("REFERENTIAL INTEGRITY CHECKS")
print("=" * 55)

fi = spark.read.table(f"{catalog}.{gold}.fact_inspection")
de = spark.read.table(f"{catalog}.{gold}.dim_establishment")
dd = spark.read.table(f"{catalog}.{gold}.dim_date")
fiv = spark.read.table(f"{catalog}.{gold}.fact_inspection_violation")
dv = spark.read.table(f"{catalog}.{gold}.dim_violation")

orphan_est = fi.join(de, fi.establishment_key == de.establishment_key, "left_anti") \
    .filter(fi.establishment_key.isNotNull()).count()
print(f"fact -> dim_establishment orphans : {orphan_est}")

orphan_dt = fi.join(dd, fi.date_key == dd.date_key, "left_anti") \
    .filter(fi.date_key.isNotNull()).count()
print(f"fact -> dim_date orphans          : {orphan_dt}")

orphan_fiv = fiv.join(fi, fiv.inspection_key == fi.inspection_key, "left_anti").count()
print(f"fact_violation -> fact orphans    : {orphan_fiv}")

orphan_viol = fiv.join(dv, fiv.violation_key == dv.violation_key, "left_anti") \
    .filter(fiv.violation_key.isNotNull()).count()
print(f"fact_violation -> dim_viol orphans: {orphan_viol}")